## Personal vibe check
Generate the top-K book recommendations for **you** (the personal user) using a trained checkpoint and look at the actual titles.

Pipeline:
1. Load a checkpoint (`two_tower_mac.pt` for the original model, `two_tower_v2_best.pt` for the v2 with regularization + log-freq).
2. Compute your user embedding from `user_features.npy` (the personal Goodreads-derived feature vector).
3. Forward every book in the catalog through the item tower → `(1.78M, 128)` item embeddings.
4. Score every book: `score = your_user_emb · item_emb`.
5. Mask books you've already rated (from `my_books.csv`).
6. Top-K with titles and genres. This is the honest test — would *you* read these?

In [ ]:
import json
from pathlib import Path

import numpy as np
import polars as pl
import torch

from mybookrec import DATA_DIR
from mybookrec.model.towers import TwoTowerModel

# Pick which checkpoint to inspect.
CHECKPOINT_PATH = DATA_DIR.parent / 'checkpoints' / 'two_tower_v2_best.pt'
TOP_K = 20

device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Checkpoint: {CHECKPOINT_PATH}  exists={CHECKPOINT_PATH.exists()}')

In [ ]:
# Load checkpoint and reconstruct model.
ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
config = ckpt['config']
print(f"config: {config}")
print(f"trained on {ckpt.get('n_batches_trained', '?'):,} batches")
print(f"checkpoint HR@10: {ckpt.get('val_hr10')}")

model = TwoTowerModel(
    user_input_dim=config['user_input_dim'],
    item_input_dim=config['item_input_dim'],
    hidden_dims=tuple(config['hidden_dims']),
    dropout=config['dropout'],
).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()

In [ ]:
# Load your personal user features (built from your Goodreads CSV in build_user_feature.ipynb).
transformed = DATA_DIR / 'transformed'
personal_user_features_np = np.load(transformed / 'user_features.npy')  # shape (779,)
personal_user_features = torch.from_numpy(personal_user_features_np).to(device).float().unsqueeze(0)  # (1, 779)
print(f'personal user features: {tuple(personal_user_features.shape)}')

# Load item features in the same order/format as training.
book_embeddings_np = np.load(transformed / 'book_embeddings.npy')
genre_matrix_np = np.load(transformed / 'genre_matrix.npy')
pages_vec_np = np.load(transformed / 'num_pages_normalized.npy')

_emb = torch.from_numpy(book_embeddings_np).to(device).float()
_genre = torch.from_numpy(genre_matrix_np).to(device).float()
_pages = torch.from_numpy(pages_vec_np).to(device).float().reshape(-1, 1)
del book_embeddings_np, genre_matrix_np, pages_vec_np
item_features = torch.cat([_emb, _genre, _pages], dim=1).contiguous()
del _emb, _genre, _pages
print(f'item_features: {tuple(item_features.shape)}')

In [ ]:
# Forward pass: your user embedding + all item embeddings in the shared space.
with torch.no_grad():
    user_emb = model.encode_user(personal_user_features)  # (1, 128)

    item_embs_list = []
    for i in range(0, item_features.shape[0], 8192):
        item_embs_list.append(model.encode_item(item_features[i:i+8192]))
    all_item_embs = torch.cat(item_embs_list, dim=0)
    del item_embs_list

    # Cosine similarity (both are L2-normalized) — no temperature scaling at inference.
    scores = (user_emb @ all_item_embs.T).squeeze(0)  # (n_books,)

print(f'scores: shape={tuple(scores.shape)}  range=[{scores.min().item():.3f}, {scores.max().item():.3f}]')

In [ ]:
# Mask books you've already rated — we don't want to recommend things you've read.
with open(transformed / 'book_id_to_index.json') as f:
    book_id_to_index = json.load(f)

my_books = pl.read_csv(transformed / 'my_books.csv')
my_book_idxs = [
    book_id_to_index[str(bid)]
    for bid in my_books['book_id'].to_list()
    if str(bid) in book_id_to_index
]
scores[my_book_idxs] = float('-inf')
print(f'masked {len(my_book_idxs)} books you already rated')

In [ ]:
# Get top-K. Then look up titles/genres from the books parquet.
top_scores, top_idxs = torch.topk(scores, k=TOP_K)
top_idxs_list = top_idxs.cpu().tolist()

# Reverse the book_id_to_index lookup so we can go top_idx -> book_id.
index_to_book_id = {v: k for k, v in book_id_to_index.items()}
top_book_ids = [index_to_book_id[i] for i in top_idxs_list]

# Pull metadata from books_with_genres.parquet (filtered to just our top-K).
meta = (
    pl.scan_parquet(transformed / 'books_with_genres.parquet')
    .select('book_id', 'title', 'num_pages', 'average_rating', 'genres')
    .filter(pl.col('book_id').is_in(top_book_ids))
    .collect()
)

# Reorder to match top-K ranking.
meta_dict = {row['book_id']: row for row in meta.to_dicts()}

print(f'{"rank":<5} {"score":<8} {"avg":<6} {"pages":<6} {"title"}')
print('-' * 100)
for rank, (bid, score) in enumerate(zip(top_book_ids, top_scores.cpu().tolist()), 1):
    row = meta_dict.get(bid, {})
    title = (row.get('title') or '?')[:80]
    avg = row.get('average_rating', '?')
    pages = row.get('num_pages') or '?'
    print(f'{rank:<5} {score:<8.4f} {str(avg)[:5]:<6} {str(pages)[:5]:<6} {title}')